In [86]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error , r2_score



# LOADING DATA
data = pd.read_csv('iot_telemetry_data.csv')
print(data.head())



             ts             device        co   humidity  light       lpg  \
0  1.594512e+09  b8:27:eb:bf:9d:51  0.004956  51.000000  False  0.007651   
1  1.594512e+09  00:0f:00:70:91:0a  0.002840  76.000000  False  0.005114   
2  1.594512e+09  b8:27:eb:bf:9d:51  0.004976  50.900000  False  0.007673   
3  1.594512e+09  1c:bf:ce:15:ec:4d  0.004403  76.800003   True  0.007023   
4  1.594512e+09  b8:27:eb:bf:9d:51  0.004967  50.900000  False  0.007664   

   motion     smoke       temp  
0   False  0.020411  22.700000  
1   False  0.013275  19.700001  
2   False  0.020475  22.600000  
3   False  0.018628  27.000000  
4   False  0.020448  22.600000  


In [60]:
data['ts'] = pd.to_datetime(data['ts'])

In [61]:
# CLEANING DATA
nan_count = data.isnull().sum()
print(nan_count)

data.drop_duplicates(inplace=True)

ts          0
device      0
co          0
humidity    0
light       0
lpg         0
motion      0
smoke       0
temp        0
dtype: int64


In [62]:
from sklearn.preprocessing import LabelEncoder
data['hour'] = data['ts'].dt.hour
data['day'] = data['ts'].dt.day

# encoding device
le = LabelEncoder()
data['device'] = le.fit_transform(data['device'])



In [68]:

data['risk_score'] = (
    0.3*data['co']+
    0.3*data['lpg']+
    0.2*data['smoke']+
    0.1*data['temp']+
    0.1*data['humidity']
)
x = data.drop(['risk_score','ts'], axis=1)
y = data['risk_score']
print(x)

        device        co   humidity  light       lpg  motion     smoke  \
0            2  0.004956  51.000000  False  0.007651   False  0.020411   
1            0  0.002840  76.000000  False  0.005114   False  0.013275   
2            2  0.004976  50.900000  False  0.007673   False  0.020475   
3            1  0.004403  76.800003   True  0.007023   False  0.018628   
4            2  0.004967  50.900000  False  0.007664   False  0.020448   
...        ...       ...        ...    ...       ...     ...       ...   
405179       0  0.003745  75.300003  False  0.006247   False  0.016437   
405180       2  0.005882  48.500000  False  0.008660   False  0.023301   
405181       1  0.004540  75.699997   True  0.007181   False  0.019076   
405182       0  0.003745  75.300003  False  0.006247   False  0.016437   
405183       2  0.005914  48.400000  False  0.008695   False  0.023400   

             temp  hour  day  
0       22.700000     0    1  
1       19.700001     0    1  
2       22.600000 

In [69]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.4)



LinearRegression()

In [82]:
# LINEAR REGRESSION MODEL

linear_model =LinearRegression()
linear_model.fit(x_train, y_train)

LinearRegression()

In [81]:
# K MEANS CLUSTERING
k_means_model = KMeans(n_clusters=3)
k_means_model.fit(x_train, y_train)

KMeans(n_clusters=3)

In [80]:
# DECISSION TREE
from sklearn.tree import DecisionTreeRegressor
d_tree_model= DecisionTreeRegressor()
d_tree_model.fit(x_train, y_train)

DecisionTreeRegressor()

In [83]:
# MAE CALCULATION
linear_y_pred = linear_model.predict(x_test)
kmeans_y_pred = k_means_model.predict(x_test)
d_tree_y_pred = d_tree_model.predict(x_test)

print("MEAN ABSOLUTE ERROR")
print(mean_absolute_error(y_test, y_pred))
print(mean_absolute_error(y_test, kmeans_y_pred))
print(mean_absolute_error(y_test, d_tree_y_pred))

MEAN ABSOLUTE ERROR
7.620875802294587
6.984165512232947
0.00037838613882827237


In [85]:
# RMSE CALCULATION
import numpy as np
print("ROOT MEAN SQUARED ERROR")
print(np.sqrt(mean_squared_error(y_test, linear_y_pred)))
print(np.sqrt(mean_squared_error(y_test, kmeans_y_pred)))
print(np.sqrt(mean_squared_error(y_test, d_tree_y_pred)))

ROOT MEAN SQUARED ERROR
3.67056923945028e-15
7.176048440560282
0.009541441856390618


In [92]:
def evaluate_model(name,model):
    pred = linear_model.predict(x_test)

    mean = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    linear_r2 = r2_score(y_test, pred)

    print("R2 SCORE")
    print(linear_r2)